# KTH LRCN Temporal Clipping - Updated Replica

This notebook is a separate replica of the current temporal clipping experiment. It explains the purpose of the code, the logic behind the frame sampling, and how the extracted frames are used in an LRCN workflow.

The goal is to convert a raw KTH video into a compact sequence of selected frames that represent the motion of an action.

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

## 1. Temporal sampling

Instead of using all frames from a video, we select a fixed number of frames evenly spaced over time. This helps reduce computation and keeps the sequence manageable for the model.

Here, we choose 16 frames and take every 6th frame.

In [ ]:
num_frames = 16
frame_step = 6
sample_indices = np.arange(
    0,
    num_frames * frame_step,
    frame_step
)

print(sample_indices)

## 2. Load the KTH video

We load one sample video from the walking action folder and read frames only at the selected indices.

In [ ]:
from pathlib import Path

base_dir = Path(r"C:\NNDL Lab Proj\KTH")
dataset_path = base_dir / "Dataset" if (base_dir / "Dataset").exists() else base_dir
video_path = dataset_path / "walking" / "person01_walking_d1_uncomp.avi"
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise FileNotFoundError(f"Video not found: {video_path}")

frames = []

for index in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ret, frame = cap.read()

    if ret:
        frames.append(frame)

cap.release()

print("Total frames loaded:", len(frames))
print("Frame shape:", frames[0].shape)

## 3. Why this works

Each selected frame is part of the same video sequence, so the list of frames represents the motion of the action over time. This makes the data suitable for LRCN-style temporal learning.

The model can see changes across frames instead of just a single image.

In [ ]:
plt.figure(figsize=(12, 8))

for i, frame in enumerate(frames):
    plt.subplot(4, 4, i + 1)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.imshow(frame_rgb)
    plt.title(f"Frame {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Interpretation

This notebook is a basic preprocessing step in the KTH action recognition pipeline. It demonstrates how a video can be reduced to a short temporal clip before being fed to a recurrent or LRCN-based model.

In practical use, this process is repeated for every video in the dataset, and all clips are labeled according to their action class before training.